# Tutorial 3: Reading the KAIROS clock

The clock combines four dials into a tripwire level and a lead time. This
notebook walks through the decision rules with made-up numbers.

**KAIROS recommends; humans decide.**

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:  # plots are optional
    plt = None

rng = np.random.default_rng(13)

## Jonas: an asymmetric loss

A missed catastrophe costs `λ` times a false alarm. The expected-loss
minimising rule is to raise a dial's alarm when `P(danger) > 1/(1+λ)`.
`λ` is published in YAML, never hard-coded.

In [ ]:
lam = 9.0
threshold = 1 / (1 + lam)
p_danger = {"acceleration": 0.07, "recursion": 0.04, "vinge": 0.22, "jonas": 0.31}
states = {dial: ("danger" if p > threshold else "calm") for dial, p in p_danger.items()}
print(f"threshold = {threshold:.2f}")
for dial, s in states.items():
    print(f"  {dial:13s} P = {p_danger[dial]:.2f} -> {s}")

## Tripwire level

In [ ]:
LEVELS = {
    0: ("Normal", "Weekly publication"),
    1: ("Watch", "More data collection, public notice"),
    2: ("Warning", "Mandatory third-party evaluation before new frontier deployments"),
    3: ("Alarm", "Pause scaling past the current frontier until H catches up"),
    4: ("Kairos", "The last window in which intervention is still cheap"),
}


def tripwire(states, confirmed_metrics=None):
    n = sum(s == "danger" for s in states.values())
    confirmed_metrics = confirmed_metrics or {}
    if n == 1 and any(confirmed_metrics.get(d, 0) >= 3 for d, s in states.items() if s == "danger"):
        n = 2
    return n


level = tripwire(states)
name, response = LEVELS[level]
print(f"Level {level}: {name}. Recommended: {response}")

## Collingridge: lead time

Lead time is the number of months until the **upper** quantile of the forecast
crosses the danger threshold. Using the upper quantile is the Jonas rule
applied to time. The interval comes from the forecast ensemble.

In [ ]:
months = np.arange(1, 121)
ensemble = 0.02 * months[None, :] * rng.lognormal(0, 0.35, size=(500, 1))  # gap growth paths
threshold_gap = 1.0
q95 = np.quantile(ensemble, 0.95, axis=0)
lead = months[np.argmax(q95 > threshold_gap)]
crossing = np.array([months[np.argmax(p > threshold_gap)] if (p > threshold_gap).any() else 120 for p in ensemble])
lo, hi = np.quantile(crossing, [0.05, 0.95])
print(f"Lead time: {lead} months (90% CI: {lo:.0f}-{hi:.0f})")

## Rawls: maximin next to expected value

The decision taken in the worst plausible scenario is reported next to the
expected-value decision, so readers see both.

In [ ]:
worst = int(np.quantile(crossing, 0.05))
print(f"expected-value view: cross in ~{crossing.mean():.0f} months")
print(f"maximin view: plan for {worst} months")